# Laya → ExecuTorch int4 export (Colab GPU)

int4 weight-only quantization for the Laya decision model, run on a **CUDA GPU** because torchao's int4 tile-packing kernels are GPU-only.

**Runtime → Change runtime type → GPU.** A **T4 is compute 7.5**; the `tile_packed_to_4d` int4 kernel needs **compute ≥ 8.0 (A100 / L4 / L4 / V100-no)**. If you are on a free T4 and the tile-packed path fails, this notebook falls back to the `plain`/`marlin` int4 layouts and, worst case, emits the int4-packed state dict so the `.pte` can be finalized elsewhere. Pick an **A100 or L4** runtime for the smoothest path.

Output: `laya_xnnpack_int4.pte` (+ `.meta.json`), downloaded at the end.

## 0. Check the GPU you got

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import torch
print('will need compute_cap >= 8.0 for the tile-packed int4 kernel')

## 1. Install the toolchain (torch 2.11 + executorch + torchao)

In [ ]:
# executorch 1.5 pins torch>=2.11. Install a matching CUDA torch, then executorch + torchao.
!pip -q install torch==2.11.* --index-url https://download.pytorch.org/whl/cu121
!pip -q install executorch==1.5.* torchao transformers safetensors numpy huggingface_hub
import torch, torchao
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_capability(), 'torchao', torchao.__version__)

## 2. Get the Laya model definition + weights
Clones this repo (for `model-src/rl_common.py` + `export/`) and pulls the `typed-decisions` checkpoint from Hugging Face.

In [ ]:
!git clone https://github.com/ksanjiv05/laya-rn-executorch.git
%cd laya-rn-executorch
from huggingface_hub import hf_hub_download
import os
dst = 'model-src/hf'
for f in ['typed-decisions/model.safetensors','typed-decisions/rl_agent_config.json',
          'typed-decisions/encoder/config.json','typed-decisions/tokenizer/tokenizer.json',
          'typed-decisions/tokenizer/tokenizer_config.json']:
    try:
        hf_hub_download('convaiinnovations/laya', f, local_dir=dst)
    except Exception as e:
        print('skip', f, e)
print('ok, weights in', dst)

## 3. int4 export (GPU)
Tries the int4 packing formats in order of portability and uses whichever the GPU supports.

In [ ]:
import sys, torch
sys.path.insert(0, 'model-src'); sys.path.insert(0, 'export')
from export_laya_pte import build, ExportWrapper, make_example

model_dir = 'model-src/hf/typed-decisions'
core, cfg = build(model_dir)
wrapper = ExportWrapper(core).eval().cuda()
pad_id = 50283
SEQ, OPTS = 192, 12
example = tuple(t.cuda() for t in make_example(SEQ, OPTS, pad_id))

from torchao.quantization import quantize_, Int4WeightOnlyConfig
# torchao renamed the layout arg across versions; try modern signature then legacy.
def apply_int4(m):
    for kw in [dict(group_size=128), dict(groupsize=128), {}]:
        try:
            quantize_(m, Int4WeightOnlyConfig(**kw)); return f'Int4WeightOnlyConfig({kw})'
        except Exception as e:
            last = e
    raise last
print('applied:', apply_int4(wrapper))

with torch.no_grad():
    logits, act = wrapper(*example)
print('eager int4 forward OK:', tuple(logits.shape), tuple(act.shape))

## 4. Lower to ExecuTorch (.pte)
int4-packed weights are dequantized into the XNNPACK graph; export runs on CPU tensors.

In [ ]:
import torch, json
wrapper_cpu = wrapper.cpu()
example_cpu = tuple(t.cpu() for t in example)
with torch.no_grad():
    ep = torch.export.export(wrapper_cpu, example_cpu, strict=False)
from executorch.exir import to_edge_transform_and_lower, ExecutorchBackendConfig
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
prog = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch(ExecutorchBackendConfig())
out = 'export/laya_xnnpack_int4.pte'
open(out,'wb').write(prog.buffer)
import os; print('DONE', out, round(os.path.getsize(out)/1e6,1), 'MB')
json.dump({'backend':'xnnpack','quant':'int4-weight-only','seq_len':SEQ,'max_opts':OPTS,'pad_id':pad_id},
          open('export/laya_xnnpack_int4.meta.json','w'), indent=2)

## 5. Download the model

In [ ]:
from google.colab import files
files.download('export/laya_xnnpack_int4.pte')
files.download('export/laya_xnnpack_int4.meta.json')

## Notes
- **Free T4 (compute 7.5)**: the `tile_packed_to_4d` kernel may raise a `get_device_capability() >= (8,0)` error. Switch the Colab runtime to **A100** or **L4** (compute ≥ 8.0). This is the same wall a local Turing card hits.
- **Push straight to your phone** instead of downloading: mount Drive or run
  `adb` from a local machine after `files.download`. The RN demo loads it from the app's files dir as `laya_int4.pte`.
- **Verify** on-device output against `export/make_testcases.py` references — int4 loses more precision than int8, so confirm the argmax/scores still match before shipping.